# Pluto M2 on Google Colab: continue speech training, evaluate, demo

Continues the M2 runs started on Kaggle (`notebooks/m2_kaggle.ipynb`). Everything is restored from the
private Hugging Face repo (`<you>/pluto-runs`): the speech dataset and the run checkpoints.

Runtime → Change runtime type → **T4 GPU**. Free Colab has **one** GPU, so the aligned and baseline
runs are resumed **one after the other**; both stop and save cleanly at `train.max_wall_time_s`,
and a disconnect loses at most the steps since the last checkpoint (every 500 steps).

Secrets (🔑 icon in the left sidebar, enable "Notebook access"):
- `HF_TOKEN` (**required**): Hugging Face write token; the runs and speech data live in your private repo.
- `GITHUB_TOKEN`: only for the final results push (and cloning while the repo is private).

Off-the-shelf components (disclosed): Kokoro-82M TTS (data synthesis), Kyutai Mimi codec, Whisper small.en.

## Setup (re-run cells 1–4 after every disconnect)

In [1]:
import os, subprocess
REPO = "thedeepakreddy/Pluto-LLM"
BRANCH = "main"

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

gh = secret("GITHUB_TOKEN")
url = f"https://{gh}@github.com/{REPO}.git" if gh else f"https://github.com/{REPO}.git"
WORK = "/content/Pluto-LLM"   # absolute: re-running this cell never nests folders
if not os.path.exists(WORK):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, WORK], check=True)
else:
    subprocess.run(["git", "-C", WORK, "fetch", "-q", url, BRANCH], check=False)
    subprocess.run(["git", "-C", WORK, "reset", "-q", "--hard", "FETCH_HEAD"], check=False)
%cd {WORK}
!apt-get install -y -q espeak-ng > /dev/null
# Same isolated Python 3.12 env as on Kaggle (Kokoro needs Python <= 3.12).
!pip install -q uv
if not os.path.exists("/content/py312/bin/python"):
    !uv venv -q --python 3.12 /content/py312
!uv pip install -q --python /content/py312/bin/python -e ".[speech,hub]"
!uv pip install -q --python /content/py312/bin/python "en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl"
LLM = "/content/py312/bin/llm"
!{LLM} --help > /dev/null && echo "llm CLI ready (Python 3.12)"
!pip install -q -e ".[hub]"
import sys, importlib, glob
sys.path.insert(0, os.path.abspath("src"))
importlib.invalidate_caches()

hf = secret("HF_TOKEN")
HUB = ""
if hf:
    os.environ["HF_TOKEN"] = hf
    from huggingface_hub import HfApi
    HUB = HfApi().whoami()["name"] + "/pluto-runs"
print("Hub sync:", HUB or "OFF: add the HF_TOKEN secret, this notebook needs it")
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content/Pluto-LLM
llm CLI ready (Python 3.12)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for llm (pyproject.toml) ... done
Hub sync: thedeepakreddy/pluto-runs
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
IDENTITY_DATASET = "thedeepakreddy/pluto-creator-identity"  # private HF dataset ("" = off)
ID = f"--set identity.hf_dataset={IDENTITY_DATASET} --set identity.speech_every=100" if IDENTITY_DATASET else ""

# Run names printed on Kaggle (`!ls runs/`). Fill these in.
ALIGNED_RUN = "runs/m2-aligned-20261102-213616-72fb023"
BASELINE_RUN = "runs/m2-baseline-20261102-213616-72fb023"
WARM = "configs/m2-warmstart.yaml"
ALIGNED = "configs/m2-speech-aligned.yaml"
BASELINE = "configs/m2-speech-baseline.yaml"

In [3]:
# Speech dataset (built on Kaggle) from the private Hub repo.
SPEECH_DIR = "speech/tinystories-speech-v1"
from huggingface_hub import snapshot_download
if not os.path.exists(f"{SPEECH_DIR}/manifest.json"):
    snapshot_download(HUB, allow_patterns=[SPEECH_DIR + "/*"], local_dir=".")
!ls {SPEECH_DIR}/manifest.json

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 32 files:   0%|          | 0/32 [00:00<?, ?it/s]

speech/tinystories-speech-v1/manifest.json


In [4]:
# Text data, tokenizer and shards (identical to M1/Kaggle; ~15-40 min) and the identity files.
!{LLM} prepare-data {WARM}
!{LLM} train-tokenizer {WARM}
!{LLM} tokenize {WARM}
if IDENTITY_DATASET:
    !{LLM} prepare-identity {ALIGNED} {ID}


data/tinystories-v1
tokenizers/tinystories-bpe-v1
shards/tinystories-shards-v1

identity/pcid-v1




## Stage 3a: resume the aligned run
`resume` uses the run's own saved config (identity and Hub sync included) and continues from its latest
checkpoint. Re-run this cell after a disconnect (after cells 1–4) until it prints `stopped: done`.

In [5]:
for run in (ALIGNED_RUN, BASELINE_RUN):   # pull both now, so a later disconnect costs nothing
    if not os.path.exists(f"{run}/config.yaml"):
        !{LLM} pull-run {HUB} {os.path.basename(run)}
!{LLM} resume {ALIGNED_RUN}


runs/m2-aligned-20261102-213616-72fb023



runs/m2-baseline-20261102-213616-72fb023


run: /content/Pluto-LLM/runs/m2-aligned-20261102-213616-72fb023
stopped: max_steps at step 6000


## Stage 3b: resume the baseline run

In [6]:
!{LLM} resume {BASELINE_RUN}

run: /content/Pluto-LLM/runs/m2-baseline-20261102-213616-72fb023
stopped: max_steps at step 6000


## Stage 4: evaluation, demo audio, report
`eval-speech` scores 200 held-out test chunks with Whisper (WER/CER, duration ratio, RTF);
`--floor` also scores an untrained model so the numbers have a reference point.

In [ ]:
!{LLM} eval-speech {ALIGNED_RUN} --floor
!{LLM} eval-speech {BASELINE_RUN} --floor


















[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/content/py312/lib/python3.12/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step

In [ ]:
from IPython.display import Audio, Markdown, display
from llm.speech.evaluate import DEMO_SENTENCES
for run in (ALIGNED_RUN, BASELINE_RUN):
    os.makedirs(f"{run}/demo", exist_ok=True)
    for i, text in enumerate(DEMO_SENTENCES[:3]):
        wav = f"{run}/demo/demo_{i}.wav"
        !{LLM} speak {run} --text "{text}" --out {wav} --seed 1
        print(run, "|", text)
        display(Audio(wav))

In [ ]:
for run in (ALIGNED_RUN, BASELINE_RUN):
    open(f"{run}/limitations.md", "w").write(
        "# Known limitations\n\n"
        "- ~30M parameters; speech is synthetic (one Kokoro voice) reading TinyStories only.\n"
        "- Audio quality is bounded by the Mimi codec; WER is measured by Whisper, itself imperfect.\n"
        "- Single run per layout, single seed.\n"
        "- Training was split across Kaggle (2x T4) and Colab (T4) sessions via checkpoint resume.\n")
    !{LLM} report {run}
!{LLM} compare {ALIGNED_RUN} {BASELINE_RUN}
display(Markdown(open(f"{ALIGNED_RUN}/report.md").read().replace("](charts/", f"]({ALIGNED_RUN}/charts/")))

## Push results to GitHub (public artifacts only; identity redacted by `export_public_results`)

In [ ]:
def run_git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr + r.stdout).replace(gh, "***") if gh else r.stderr + r.stdout
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout

if not gh:
    print("GITHUB_TOKEN not set; skipping push")
else:
    from llm.publish import export_public_results   # redacts creator identity; raises on any leak
    for run in (ALIGNED_RUN, BASELINE_RUN):
        export_public_results(run, f"results/m2/{os.path.basename(run)}")
    run_git("config", "user.name", "Pluto Colab")
    run_git("config", "user.email", "colab@users.noreply.github.com")
    run_git("add", "results")
    if not run_git("status", "--porcelain", "results").strip():
        print("nothing changed; not committing")
    else:
        run_git("commit", "-m", "results(m2): run reports")
        run_git("pull", "--rebase", url, BRANCH)
        run_git("push", url, f"HEAD:{BRANCH}")
        print("pushed results/")